# Explore a survey for top-tail diagnostics

Choose one `.dta` file in `01-input/country`, then run the cells in order. The notebook does not modify survey data.

**Weights are used everywhere:** weighted quantiles define the LIS ceiling and p90 threshold; weighted means, histograms, CCDFs, and Pareto estimation represent the survey population rather than giving every sampled record equal importance.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / '01-input' / 'country').exists():
    raise FileNotFoundError('Open this notebook with bindata_check as the working directory.')

INPUT_DIR = ROOT / '01-input' / 'country'
OUTPUT_DIR = ROOT / 'top_censoring' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

available_files = sorted(path.name for path in INPUT_DIR.glob('*.dta'))
available_files

In [ ]:
# Change this value, then rerun this cell and all cells below.
SURVEY_FILE = 'MWI_1997.dta'

if SURVEY_FILE not in available_files:
    raise ValueError(f'{SURVEY_FILE} is not in 01-input/country. Choose one of: {available_files}')

df = pd.read_stata(INPUT_DIR / SURVEY_FILE)
print(f'Loaded {SURVEY_FILE}: {len(df):,} rows and {len(df.columns):,} columns')
display(df.head())
print('Columns:')
print(', '.join(df.columns))

In [ ]:
def choose_weight_column(columns):
    for candidate in ['weight_p', 'weight', 'weight_h']:
        if candidate in columns:
            return candidate
    raise ValueError('No recognized weight column found.')

def weighted_quantile(values, weights, probabilities):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]
    cumulative_weight = np.cumsum(weights) / weights.sum()
    return np.array([values[np.searchsorted(cumulative_weight, p, side='left')] for p in probabilities])

weight_column = choose_weight_column(df.columns)
required = ['welfare', weight_column, 'cpi2021', 'icp2021']
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['welfare_ppp_day'] = df['welfare'] / (df['cpi2021'] * df['icp2021'] * 365)
analysis = df.loc[
    np.isfinite(df['welfare_ppp_day']) & (df['welfare_ppp_day'] > 0) &
    np.isfinite(df[weight_column]) & (df[weight_column] > 0),
    ['welfare_ppp_day', weight_column]
].copy()
analysis = analysis.rename(columns={weight_column: 'weight'})

print('Weight column:', weight_column)
print('Valid records:', f'{len(analysis):,}')
print('Population represented:', f"{analysis['weight'].sum():,.0f}")

In [ ]:
x = analysis['welfare_ppp_day'].to_numpy()
w = analysis['weight'].to_numpy()
q01, q25, q50, q75, q90, q95, q99 = weighted_quantile(x, w, [0.01, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

summary = pd.DataFrame({
    'statistic': ['p1', 'p25', 'median', 'p75', 'p90', 'p95', 'p99', 'weighted mean', 'maximum'],
    'welfare_ppp_day': [q01, q25, q50, q75, q90, q95, q99, np.average(x, weights=w), x.max()]
})
display(summary.style.format({'welfare_ppp_day': '{:,.3f}'}))

In [ ]:
# LIS top ceiling is computed on log welfare.
q1_log, q3_log = weighted_quantile(np.log(x), w, [0.25, 0.75])
lis_ceiling = np.exp(q3_log + 3 * (q3_log - q1_log))
above_lis = x > lis_ceiling

print(f'LIS ceiling (k=3): {lis_ceiling:,.2f} PPP USD/day')
print(f'Records above ceiling: {above_lis.sum():,}')
print(f'Weighted population share above ceiling: {w[above_lis].sum() / w.sum():.3%}')
print(f'Welfare share above ceiling: {(x[above_lis] * w[above_lis]).sum() / (x * w).sum():.3%}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(x, bins=np.geomspace(x.min(), x.max(), 80), weights=w / w.sum(), color='#4C78A8')
ax.axvline(lis_ceiling, color='#009E73', linestyle='--', label='LIS k=3 ceiling')
ax.set_xscale('log')
ax.set_xlabel('Welfare (2021 PPP USD per person per day; log scale)')
ax.set_ylabel('Weighted population share')
ax.set_title(f'{SURVEY_FILE}: weighted welfare distribution')
ax.legend()
plt.show()

In [ ]:
# Pareto fit to the top 10% of the weighted distribution.
p90 = weighted_quantile(x, w, [0.90])[0]
in_tail = x >= p90
tail_x = x[in_tail]
tail_w = w[in_tail]
pareto_alpha = tail_w.sum() / np.sum(tail_w * np.log(tail_x / p90))

order = np.argsort(tail_x)
tail_x = tail_x[order]
tail_w = tail_w[order]
empirical_ccdf = np.cumsum(tail_w[::-1])[::-1] / tail_w.sum()
pareto_ccdf = (p90 / tail_x) ** pareto_alpha

print(f'Pareto threshold: p90 = {p90:,.2f}')
print(f'Pareto alpha: {pareto_alpha:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.step(tail_x, empirical_ccdf, where='post', color='#0072B2', label='Observed weighted tail')
ax.plot(tail_x, pareto_ccdf, color='#D55E00', label='Pareto fitted from p90')
ax.axvline(lis_ceiling, color='#009E73', linestyle='--', label='LIS k=3 ceiling')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Welfare (2021 PPP USD/day; log scale)')
ax.set_ylabel('Share of p90 tail at or above welfare')
ax.set_title(f'{SURVEY_FILE}: observed upper tail versus Pareto fit')
ax.legend()
plt.show()

In [ ]:
# Inspect the highest welfare values. Do not interpret them as errors solely from this table.
top_records = df.loc[
    np.isfinite(df['welfare_ppp_day']) & np.isfinite(df[weight_column]),
    [column for column in ['hhid', 'welfare', 'welfare_ppp_day', weight_column, 'hsize', 'urban', 'year', 'survname'] if column in df.columns]
].sort_values('welfare_ppp_day', ascending=False).head(20)
display(top_records)